In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        if member.name == "NCS" or member.name.startswith("NCS/"):
            member.name = "OmniAligner" + member.name[3:]
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "OmniAligner"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%cd {PACKAGE_ROOT}


## Download and run
Use a **fresh Python 3.10/3.11 Jupyter kernel** with `wget` and `git` available. The first two commands download the public data archive and clone the `rev1` branch. No private directory or credentials are required. The six-dataset scoring run is CPU-based; allow 48–64 GB RAM for the largest dataset. The table-only and DLPFC examples are smaller.


# OmniAligner: reproduce the selected NCS results

This notebook recomputes the six reported metrics from **original expression/annotations + selected NPZ coordinates**. No aligned H5AD export is needed. The selected weights are verified and archived alongside the coordinates; scoring does not refit the model or replay the optimiser.

The reference CSV is used only for the final comparison, never to generate recomputed scores. Six datasets, the original adjacent pairs and the original evaluation gene arguments are retained. These are selected development results, with the original annotation-assisted evaluation limitations.

The release data contain exact original expression values for the frozen evaluation-gene panel, original cell IDs, annotations and raw coordinates, exported losslessly as sparse NPZs. They are sufficient for the scoring computation; full H5AD downloads are linked separately. Input hashes bind the exported arrays to the audited source H5ADs. No benchmark-server directory is required.

## 1. Load the downloaded benchmark inputs
The paths below are relative to the cloned package. All 18 MLN pairs are included. Each dataset is saved after scoring; reference values never fill a missing or failed calculation.

In [1]:
from pathlib import Path
import os, sys, json, csv, hashlib, importlib.util, importlib.metadata, warnings, time
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from IPython.display import display

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
INPUT_ROOT = DATA_ROOT
input_records = json.loads((INPUT_ROOT / "evaluation_inputs_manifest.json").read_text())
input_index = {(x["dataset"], x["layer"]): x for x in input_records}
OUTPUT_ROOT = REPO_ROOT / "results" / "omni"
DATASETS = ["DLPFC", "MHPR", "MMPC", "MPC", "E14_16", "MLN"]
VERIFY_SOURCE_HASHES = True
ATOL = 1e-10
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
manifest = json.loads((REPO_ROOT / "notebooks/OmniAligner_NCS_manifest.json").read_text())
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)
reference_path = REPO_ROOT / "notebooks/OmniAligner_NCS_reference.csv"
assert sha256(reference_path) == manifest["expected_table_sha256"]
reference = pd.read_csv(reference_path).set_index("Dataset")
print("Reference values (not recomputed yet)")
display(reference[METRICS].round(6))

Reference values (not recomputed yet)


,PCC,CosSim,SSIM,MI,MSE,Region
Dataset,,,,,,
DLPFC,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759
MHPR,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039
MMPC,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546
MPC,0.333290,0.625865,0.323609,0.152123,0.341752,0.978106
E14_16,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624
MLN,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042


## 2. Verify inputs and the frozen scorer
Hashes bind the exact source H5ADs, selected coordinates, nine checkpoints and original scoring implementation. The six metric directions are PCC/CosSim/SSIM/MI/Region ↑ and MSE ↓. MSE scales differ across datasets; no cross-dataset mean is calculated.

In [2]:
helper_dir = REPO_ROOT / manifest["helper_dir"]
for name, expected_hash in manifest["helper_sha256"].items():
    assert sha256(helper_dir / name) == expected_hash, name
sys.path.insert(0, str(helper_dir))
import ncs_score as scoring
native_path = REPO_ROOT / manifest["native_scoring_file"]
assert sha256(native_path) == manifest["native_scoring_sha256"]
spec = importlib.util.spec_from_file_location("selected_ncs_native_metrics", native_path)
native = importlib.util.module_from_spec(spec)
spec.loader.exec_module(native)

inventory = []
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    assert sha256(DATA_ROOT / config["coordinates"]) == config["coordinates_sha256"], dataset
    for item in config["weights"]:
        assert sha256(DATA_ROOT / item["path"]) == item["sha256"], item["path"]
    for layer in range(len(config["sections"])):
        record = input_index[(dataset, layer)]
        source = INPUT_ROOT / record["path"]
        assert source.is_file(), source
        assert record["source_h5ad_sha256"] == config["source_sha256"][layer]
        if VERIFY_SOURCE_HASHES:
            assert sha256(source) == record["sha256"], source
    inventory.append({"Dataset": dataset, "Sections": len(config["sections"]), "Pairs": config["pairs"],
                      "Original cells": config["original_cells"], "Scored cells": config["evaluation_cells"],
                      "Gene arguments": len(config["evaluation_genes"]), "Unique genes": len(set(config["evaluation_genes"]))})
display(pd.DataFrame(inventory).set_index("Dataset"))
save_json(OUTPUT_ROOT / "INPUT_VALIDATION.json", {"source_hashes_checked": VERIFY_SOURCE_HASHES, "datasets": DATASETS, "inventory": inventory})

,Sections,Pairs,Original cells,Scored cells,Gene arguments,Unique genes
Dataset,,,,,,
DLPFC,4,3,14243,14243,123,117
MHPR,5,4,28317,28317,155,155
MMPC,3,2,3190,3190,100,100
MPC,3,2,5257,5250,79,79
E14_16,16,15,15295,15150,100,100
MLN,19,18,1097769,1097769,100,100


## 3. Join selected coordinates to original cells
The join uses `s<section index>_<original obs_name>`, with an exact one-to-one identity check. For MPC and E14–16 the saved `historical_cohort` mask reproduces the reported common-cell evaluation; the additional original cells are not silently included. DLPFC preserves 123 gene arguments (117 distinct genes), including the historical multiplicity. All other gene orders are also unchanged.

Coordinates remain in the saved benchmark frame, without rescaling or a physical-micrometre claim. Expression is neither normalised nor otherwise altered here.

In [3]:
def load_scoring_slice(dataset, layer, coordinates):
    config = manifest["datasets"][dataset]
    record = input_index[(dataset, layer)]
    with np.load(INPUT_ROOT / record["path"], allow_pickle=False) as raw:
        matrix = sp.csr_matrix((raw["data"], raw["indices"], raw["indptr"]), shape=tuple(raw["shape"]))
        source = ad.AnnData(X=matrix)
        source.obs_names = raw["ids"].astype(str)
        source.var_names = raw["genes"].astype(str)
        source.obs[config["label_column"]] = raw["labels"].astype(str)
        source.obsm["spatial"] = raw["raw_xy"]
    try:
        wanted = [f"s{layer}_{value}" for value in source.obs_names]
        selected = coordinates["layers"] == layer
        ids = coordinates["ids"][selected].tolist()
        assert len(set(ids)) == len(ids) == len(wanted)
        assert set(ids) == set(wanted), (dataset, layer, "Cell identity mismatch")
        lookup = {value: index for index, value in enumerate(ids)}
        order = np.asarray([lookup[value] for value in wanted])
        xy = coordinates["nonrigid"][selected][order]
        keep = coordinates["historical_cohort"][selected][order] if "historical_cohort" in coordinates.files else np.ones(source.n_obs, dtype=bool)
        assert np.isfinite(xy).all() and xy.shape == (source.n_obs, 2)
        unique_genes = list(dict.fromkeys(config["evaluation_genes"]))
        data = source[keep, unique_genes].copy()
        data.obsm["spatial"] = xy[keep]
        data.obs["Region"] = data.obs[config["label_column"]].astype(str)
        if not sp.issparse(data.X):
            data.X = sp.csr_matrix(data.X)
        return data
    finally:
        if source.isbacked:
            source.file.close()

## 4. Recompute every adjacent pair
The original `score_pair` function uses a 10 × 10 overlap grid and the original native metric functions, with random seed 0 where required. It retains undefined metrics, errors and native fallback diagnostics. Gene means and then adjacent-pair means use the original strict averaging function: missing values are never dropped. Recalculation always runs; no cached reference scores are used.

In [4]:
recomputed_rows = []
started = time.time()
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    target = OUTPUT_ROOT / dataset
    target.mkdir(exist_ok=True)
    pair_rows, gene_rows = [], []
    with np.load(DATA_ROOT / config["coordinates"], allow_pickle=False) as coordinates:
        assert len(coordinates["ids"]) == config["original_cells"]
        assert set(coordinates["layers"].tolist()) == set(range(len(config["sections"])))
        reference_slice = load_scoring_slice(dataset, 0, coordinates)
        for pair_index in range(config["pairs"]):
            moving_slice = load_scoring_slice(dataset, pair_index + 1, coordinates)
            values, gene_records, support = scoring.score_pair(native, reference_slice, moving_slice, config["evaluation_genes"])
            assert [row["gene"] for row in gene_records] == config["evaluation_genes"]
            pair_rows.append({"Dataset": dataset, "pair_index": pair_index, **values, **support})
            gene_rows.extend({"pair_index": pair_index, "gene_argument_index": index, **row} for index, row in enumerate(gene_records))
            pd.DataFrame(pair_rows).to_csv(target / "pair_metrics.csv", index=False)
            save_json(OUTPUT_ROOT / "PROGRESS.json", {"dataset": dataset, "pairs_done": pair_index + 1, "pairs_expected": config["pairs"]})
            reference_slice = moving_slice
    assert len(pair_rows) == config["pairs"]
    means = {display_name: scoring.c.strict_mean([row[native_key] for row in pair_rows], config["pairs"])
             for display_name, native_key in zip(METRICS, NATIVE_KEYS)}
    pd.DataFrame(gene_rows).to_csv(target / "gene_metrics.csv.gz", index=False, compression="gzip")
    save_json(target / "RECOMPUTED.json", {"dataset": dataset, "pairs": config["pairs"], "metrics": means,
                                          "coordinates_sha256": config["coordinates_sha256"], "source_hashes_checked": VERIFY_SOURCE_HASHES})
    recomputed_rows.append({"Dataset": dataset, **means})
    print(f"{dataset}: {config['pairs']}/{config['pairs']} pairs recomputed")
recomputed = pd.DataFrame(recomputed_rows).set_index("Dataset")
recomputed.to_csv(OUTPUT_ROOT / "OmniAligner_NCS_recomputed.csv")
display(recomputed[METRICS].round(6))

DLPFC: 3/3 pairs recomputed


MHPR: 4/4 pairs recomputed


MMPC: 2/2 pairs recomputed


MPC: 2/2 pairs recomputed


E14_16: 15/15 pairs recomputed


MLN: 18/18 pairs recomputed


,PCC,CosSim,SSIM,MI,MSE,Region
Dataset,,,,,,
DLPFC,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759
MHPR,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039
MMPC,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546
MPC,0.333290,0.625865,0.323609,0.152123,0.341752,0.978106
E14_16,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624
MLN,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042


## 5. Compare with the current result table
Each metric is checked separately at absolute tolerance `1e-10`, with no relative tolerance. A mismatch raises an error and leaves the recomputed files available for investigation; it never overwrites the reference table.

In [5]:
expected = reference.loc[DATASETS, METRICS].astype(float)
actual = recomputed.loc[DATASETS, METRICS].astype(float)
difference = actual - expected
matches = np.isclose(actual.to_numpy(), expected.to_numpy(), atol=ATOL, rtol=0, equal_nan=False)
display(difference.rename_axis("Dataset: recomputed minus reference"))
validation = {"all_metrics_match": bool(matches.all()), "datasets": DATASETS,
              "metrics_checked": int(matches.size), "absolute_tolerance": ATOL,
              "maximum_absolute_difference": float(np.abs(difference.to_numpy()).max()),
              "elapsed_seconds": time.time() - started,
              "source_hashes_checked": VERIFY_SOURCE_HASHES,
              "packages": {name: importlib.metadata.version(name) for name in ["numpy", "scipy", "pandas", "anndata", "scanpy", "scikit-learn"]}}
save_json(OUTPUT_ROOT / "VALIDATION.json", validation)
assert matches.all(), "Recomputed metrics differ: inspect pair_metrics.csv and gene_metrics.csv.gz; the reference was not changed."
print(f"PASS: {matches.size}/{matches.size} metrics match the reference at absolute tolerance {ATOL:g}.")

,PCC,CosSim,SSIM,MI,MSE,Region
Dataset: recomputed minus reference,,,,,,
DLPFC,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.775558e-17,0.000000e+00
MHPR,1.110223e-16,1.110223e-16,0.000000e+00,0.000000e+00,4.440892e-16,-1.110223e-16
MMPC,2.775558e-17,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00
MPC,0.000000e+00,0.000000e+00,5.551115e-17,-5.551115e-17,0.000000e+00,0.000000e+00
E14_16,-5.551115e-17,-1.110223e-16,-5.551115e-17,2.775558e-17,0.000000e+00,0.000000e+00
MLN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.245005e-17,5.551115e-17


PASS: 36/36 metrics match the reference at absolute tolerance 1e-10.


## Interpretation
Successful agreement verifies reproduction of these selected saved results under their original scoring protocol. It does not establish independent anatomical validation, blind model selection, or equivalence between annotation consistency and biological fidelity. The weights, coordinates, source inputs, gene panels and cohort definitions must travel together when reproducing the results.